In [11]:
# ===============================
# Voice2Music – PerformanceRNN Rich + Multitrack (Windows-safe Fluidsynth)
# ===============================
import os
import numpy as np
import librosa
import pretty_midi
import subprocess
from pydub import AudioSegment
from note_seq import midi_file_to_note_sequence, sequence_proto_to_midi_file
from magenta.models.performance_rnn import performance_sequence_generator
from magenta.models.shared import sequence_generator_bundle
from note_seq.protobuf import generator_pb2
import soundfile as sf

# -------------------------------
# CONFIG
# -------------------------------
DATA_FOLDER = r"D:\shree\Miniproject\voicetomusic\Voice2Music\data"
OUTPUT_FOLDER = r"D:\shree\Miniproject\voicetomusic\Voice2Music\output\performance_rnn_rich_win"
os.makedirs(DATA_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

INPUT_WAV = os.path.join(DATA_FOLDER, "humming.wav")
INPUT_MIDI = os.path.join(OUTPUT_FOLDER, "input_melody.mid")
GENERATED_MIDI = os.path.join(OUTPUT_FOLDER, "performance_generated.mid")
FINAL_MIX = os.path.join(OUTPUT_FOLDER, "voice_music_rich.wav")

PERFORMANCE_BUNDLE = r"D:\shree\Miniproject\voicetomusic\Voice2Music\models\performance_rnn\performance_with_dynamics.mag"
SOUNDFONT = r"D:\shree\Miniproject\voicetomusic\Voice2Music\models\TimGM6mb.sf2"
FLUIDSYNTH_EXE = r"D:\shree\Miniproject\voicetomusic\Voice2Music\models\fluidsynth\bin\fluidsynth.exe"

# -------------------------------
# STEP 1: AUDIO → MIDI
# -------------------------------
def audio_to_midi(input_wav, output_midi):
    import crepe
    print("Extracting melody from voice...")
    y, sr = librosa.load(input_wav, sr=16000)
    time, freq, confidence, _ = crepe.predict(y, sr, viterbi=True)

    pm = pretty_midi.PrettyMIDI()
    inst = pretty_midi.Instrument(program=0)
    notes = []
    for t, f, c in zip(time, freq, confidence):
        if c > 0.5 and f > 0:
            n = int(pretty_midi.hz_to_note_number(f))
            notes.append((n, t, t + 0.25))
    if not notes:
        notes.append((60, 0.0, 0.5))

    # merge consecutive same notes
    merged = []
    cur, start, end = notes[0]
    for n, s, e in notes[1:]:
        if n == cur and s - end < 0.1:
            end = e
        else:
            merged.append((cur, start, end))
            cur, start, end = n, s, e
    merged.append((cur, start, end))

    for n, s, e in merged:
        note = pretty_midi.Note(velocity=100, pitch=n, start=s, end=e)
        inst.notes.append(note)

    pm.instruments.append(inst)
    pm.write(output_midi)
    print(f"MIDI saved: {output_midi}")
    return output_midi

# -------------------------------
# STEP 2: LOAD PerformanceRNN
# -------------------------------
def load_performance_rnn(bundle_path):
    print("Loading PerformanceRNN model...")
    bundle = sequence_generator_bundle.read_bundle_file(bundle_path)
    generator_map = performance_sequence_generator.get_generator_map()
    model = generator_map['performance_with_dynamics'](checkpoint=None, bundle=bundle)
    model.initialize()
    return model

# -------------------------------
# STEP 3: GENERATE ACCOMPANIMENT
# -------------------------------
def generate_accompaniment(input_midi, output_midi, model, duration=32.0, temperature=1.2):
    seq = midi_file_to_note_sequence(input_midi)
    if seq.total_time == 0:
        n = seq.notes.add()
        n.pitch = 60
        n.start_time = 0.0
        n.end_time = 0.5
        n.velocity = 80

    start_time = seq.total_time + 0.01
    end_time = start_time + duration

    options = generator_pb2.GeneratorOptions()
    options.args['temperature'].float_value = temperature
    options.generate_sections.add(start_time=start_time, end_time=end_time)

    print("Generating polyphonic accompaniment...")
    generated_seq = model.generate(seq, options)
    sequence_proto_to_midi_file(generated_seq, output_midi)
    print(f"Generated MIDI saved: {output_midi}")
    return output_midi

# -------------------------------
# STEP 4: SPLIT + LAYER + SYNTHESIZE (Windows-safe Fluidsynth)
# -------------------------------
def synthesize_layer_windows(inst, output_wav):
    temp_midi = output_wav.replace(".wav","_temp.mid")
    pm = pretty_midi.PrettyMIDI()
    pm.instruments.append(inst)
    pm.write(temp_midi)

    cmd = [
        FLUIDSYNTH_EXE,
        "-ni",
        "-a", "null",     # disable audio output
        "-F", output_wav, # save to WAV
        SOUNDFONT,
        temp_midi,
        "-r", "44100"
    ]
    print("Running Fluidsynth:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    os.remove(temp_midi)
    print(f"Synthesized {output_wav} successfully")

def split_layer_synthesize(input_midi, output_folder):
    pm = pretty_midi.PrettyMIDI(input_midi)
    bass = pretty_midi.Instrument(program=32)
    mid  = pretty_midi.Instrument(program=0)
    high = pretty_midi.Instrument(program=0)

    for inst in pm.instruments:
        for note in inst.notes:
            # bass down one octave
            bass.notes.append(pretty_midi.Note(velocity=np.clip(note.velocity-20,40,127),
                                               pitch=max(0,note.pitch-12),
                                               start=note.start, end=note.end))
            # mid original
            mid.notes.append(note)
            # high up one octave
            high.notes.append(pretty_midi.Note(velocity=np.clip(note.velocity-10,40,127),
                                               pitch=min(127,note.pitch+12),
                                               start=note.start, end=note.end))

    bass_wav = os.path.join(output_folder, "bass.wav")
    mid_wav  = os.path.join(output_folder, "mid.wav")
    high_wav = os.path.join(output_folder, "high.wav")

    synthesize_layer_windows(bass, bass_wav)
    synthesize_layer_windows(mid, mid_wav)
    synthesize_layer_windows(high, high_wav)

    return bass_wav, mid_wav, high_wav

# -------------------------------
# STEP 5: MIX WITH VOICE + PANNING
# -------------------------------
from pydub import AudioSegment
from pydub.effects import compress_dynamic_range, normalize, low_pass_filter, high_pass_filter

def mix_with_voice_highlighted(voice_wav, track_wavs, output_wav):
    # Load voice and tracks
    voice = AudioSegment.from_wav(voice_wav)
    bass  = AudioSegment.from_wav(track_wavs[0]).pan(-0.5) - 8  # reduce more
    mid   = AudioSegment.from_wav(track_wavs[1]).pan(0.0) - 10  # reduce more
    high  = AudioSegment.from_wav(track_wavs[2]).pan(0.5) - 8  # reduce more

    # Extend tracks to match voice length
    for t in [bass, mid, high]:
        if len(t) < len(voice):
            t += AudioSegment.silent(duration=(len(voice)-len(t)))

    # Compress voice slightly to make it more prominent
    voice = compress_dynamic_range(voice, threshold=-20.0, ratio=2.0)
    voice = normalize(voice)  # normalize after compression

    # Optional: gentle high-pass to remove low rumble
    voice = high_pass_filter(voice, 80)
    # Optional: gentle low-pass to remove harsh highs
    voice = low_pass_filter(voice, 12000)

    # Overlay tracks
    mix = voice.overlay(bass).overlay(mid).overlay(high)

    # Normalize final mix
    mix = normalize(mix)
    mix.export(output_wav, format="wav")
    print(f"Final highlighted mix saved: {output_wav}")
    return output_wav


# -------------------------------
# RUN PIPELINE
# -------------------------------
audio_to_midi(INPUT_WAV, INPUT_MIDI)
performance_rnn = load_performance_rnn(PERFORMANCE_BUNDLE)
generate_accompaniment(INPUT_MIDI, GENERATED_MIDI, performance_rnn)
bass_wav, mid_wav, high_wav = split_layer_synthesize(GENERATED_MIDI, OUTPUT_FOLDER)
mix_with_voice(INPUT_WAV, [bass_wav, mid_wav, high_wav], FINAL_MIX)

print("\nDONE! Your PerformanceRNN output is now richer, multitrack, and Windows-safe at:", FINAL_MIX)


Extracting melody from voice...
MIDI saved: D:\shree\Miniproject\voicetomusic\Voice2Music\output\performance_rnn_rich_win\input_melody.mid
Loading PerformanceRNN model...
'model_variables' collection should be of type 'byte_list', but instead is of type 'node_list'.


'model_variables' collection should be of type 'byte_list', but instead is of type 'node_list'.


INFO:tensorflow:Restoring parameters from C:\Users\All\AppData\Local\Temp\tmp_f6twjql\model.ckpt


INFO:tensorflow:Restoring parameters from C:\Users\All\AppData\Local\Temp\tmp_f6twjql\model.ckpt


Generating polyphonic accompaniment...
INFO:tensorflow:Need to generate 3199 more steps for this sequence, will try asking for 1920 RNN steps


INFO:tensorflow:Need to generate 3199 more steps for this sequence, will try asking for 1920 RNN steps


INFO:tensorflow:Beam search yields sequence with log-likelihood: -6949.924316 


INFO:tensorflow:Beam search yields sequence with log-likelihood: -6949.924316 


Generated MIDI saved: D:\shree\Miniproject\voicetomusic\Voice2Music\output\performance_rnn_rich_win\performance_generated.mid
Running Fluidsynth: D:\shree\Miniproject\voicetomusic\Voice2Music\models\fluidsynth\bin\fluidsynth.exe -ni -a null -F D:\shree\Miniproject\voicetomusic\Voice2Music\output\performance_rnn_rich_win\bass.wav D:\shree\Miniproject\voicetomusic\Voice2Music\models\TimGM6mb.sf2 D:\shree\Miniproject\voicetomusic\Voice2Music\output\performance_rnn_rich_win\bass_temp.mid -r 44100
Synthesized D:\shree\Miniproject\voicetomusic\Voice2Music\output\performance_rnn_rich_win\bass.wav successfully
Running Fluidsynth: D:\shree\Miniproject\voicetomusic\Voice2Music\models\fluidsynth\bin\fluidsynth.exe -ni -a null -F D:\shree\Miniproject\voicetomusic\Voice2Music\output\performance_rnn_rich_win\mid.wav D:\shree\Miniproject\voicetomusic\Voice2Music\models\TimGM6mb.sf2 D:\shree\Miniproject\voicetomusic\Voice2Music\output\performance_rnn_rich_win\mid_temp.mid -r 44100
Synthesized D:\shree